# Baby Step 4 — Produce the First Litigation-Committee-Ready Product

**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

This notebook converts the governed five-matter state into a committee memorandum, dashboard, reproducible exhibits, presentation deck, DEC-004, and a refreshed hot cache. Recommendation V1 remains preserved. Recommendation V2 remains prohibited.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, csv, datetime, statistics
from collections import Counter

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")
if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT/"00_System"/"Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))
if 3 not in state.get("completed_steps",[]):
    raise RuntimeError("Baby Step 3 is not complete.")

matters = json.loads((VAULT/"data"/"active_matters.json").read_text(encoding="utf-8"))
recommendations = json.loads((VAULT/"data"/"baby_step_1_recommendations_v1.json").read_text(encoding="utf-8"))
reviews = json.loads((VAULT/"data"/"baby_step_2_review_records.json").read_text(encoding="utf-8"))
claims = json.loads((VAULT/"data"/"baby_step_3_claims.json").read_text(encoding="utf-8"))
contradictions = json.loads((VAULT/"data"/"baby_step_3_contradictions.json").read_text(encoding="utf-8"))
permissions = json.loads((VAULT/"data"/"baby_step_3_permission_state.json").read_text(encoding="utf-8"))

print("Matters:",len(matters))
print("Recommendations:",len(recommendations))


## Committee classification

Each matter is assigned one bounded internal path:

- ADVANCE INTERNAL DILIGENCE
- ADVANCE WITH QUALIFICATION
- HOLD FOR CONTRADICTION RESOLUTION


In [ ]:
portfolio = {}
for matter in matters:
    mid = matter["matter_id"]
    rec = next(x for x in recommendations if x["matter_id"]==mid)
    review = next(x for x in reviews if x["matter_id"]==mid)
    perm = next(x for x in permissions if x["matter_id"]==mid)
    local_claims = [x for x in claims if x["matter_id"]==mid]
    local_cons = [x for x in contradictions if x["matter_id"]==mid]
    avg_conf = round(statistics.mean(x["confidence_score"] for x in local_claims),2)
    portfolio[mid] = {
        "matter":matter,
        "recommendation":rec,
        "review":review,
        "permission":perm,
        "claims":local_claims,
        "contradictions":local_cons,
        "average_claim_confidence":avg_conf,
        "lowest_claims":sorted(local_claims,key=lambda x:x["confidence_score"])[:3]
    }

committee_paths = []
for mid,record in portfolio.items():
    perm = record["permission"]["permission_state"]
    status = record["review"]["generalization_status"]
    if perm=="HOLD_FOR_RELIANCE":
        path = "HOLD FOR CONTRADICTION RESOLUTION"
    elif perm=="QUALIFIED_INTERNAL_USE" or status=="QUALIFY":
        path = "ADVANCE WITH QUALIFICATION"
    else:
        path = "ADVANCE INTERNAL DILIGENCE"
    committee_paths.append({
        "matter_id":mid,
        "committee_path":path,
        "recommendation_id":record["recommendation"]["recommendation_id"],
        "permission_state":perm,
        "generalization_status":status,
        "average_claim_confidence":record["average_claim_confidence"],
        "open_contradictions":len(record["contradictions"])
    })

(VAULT/"data"/"baby_step_4_committee_paths.json").write_text(
    json.dumps(committee_paths,indent=2),encoding="utf-8"
)
print(json.dumps(committee_paths,indent=2))


In [ ]:
def write_note(path,lines):
    path.write_text("\n".join(lines).strip()+"\n",encoding="utf-8")

memo = [
    "# Litigation Committee Memorandum — Baby Step 4","",
    f"**Date:** {datetime.date.today().isoformat()}","",
    "## Executive conclusion","",
    "The exo-brain completed a governed strategy-development cycle for five synthetic corporate civil matters.",
    "Recommendation V1 exists for each matter, but committee treatment depends on evidence confidence, contradiction severity, and permission state.","",
    "## Portfolio decision table","",
    "| Matter | Recommendation V1 | Confidence | Permission | Contradictions | Committee path |",
    "|---|---|---:|---|---:|---|"
]
for item in committee_paths:
    rec = portfolio[item["matter_id"]]["recommendation"]
    memo.append(
        f"| [[../02_Active_Matters/{item['matter_id']}]] | "
        f"{rec['preferred_strategy']} | {rec['confidence_score']} | "
        f"{item['permission_state']} | {item['open_contradictions']} | "
        f"**{item['committee_path']}** |"
    )

memo += ["","## Matter analyses",""]
for mid,record in portfolio.items():
    matter = record["matter"]
    rec = record["recommendation"]
    review = record["review"]
    perm = record["permission"]
    path = next(x["committee_path"] for x in committee_paths if x["matter_id"]==mid)
    memo += [
        f"### {mid} — {matter['caption']}","",
        f"**Cause of action:** {matter['cause_of_action']}  ",
        f"**Procedural posture:** {matter['procedural_posture']}  ",
        f"**Recommendation V1:** {rec['preferred_strategy']}  ",
        f"**Fallback:** {rec['fallback_strategy']}  ",
        f"**Recommendation confidence:** {rec['confidence_score']}/100  ",
        f"**Average claim confidence:** {record['average_claim_confidence']}/100  ",
        f"**Generalization status:** {review['generalization_status']}  ",
        f"**Permission state:** {perm['permission_state']}  ",
        f"**Committee path:** {path}","",
        "#### Supporting authorities",""
    ]
    memo += [f"- [[../01_Precedents/{pid}]]" for pid in rec["supporting_authorities"][:3]] or ["- None"]
    memo += ["","#### Contrary authorities",""]
    memo += [f"- [[../01_Precedents/{pid}]]" for pid in rec["contrary_authorities"][:3]] or ["- None"]
    memo += ["","#### Lowest-confidence claims",""]
    for claim in record["lowest_claims"]:
        memo.append(f"- [[../15_Claims/{claim['claim_id']}]] — {claim['confidence_score']}/100")
    memo += ["","#### Open contradictions",""]
    memo += [
        f"- [[../16_Contradictions/{c['contradiction_id']}]] — {c['severity']} — {c['category']}"
        for c in record["contradictions"]
    ] or ["- None"]
    memo += ["","#### Requested committee action","",f"**{path}**",""]

memo += [
    "## Global requested decision","",
    "Approve the five matter-specific committee paths as the bounded internal baseline.","",
    "## Not requested","",
    "- No Recommendation V2","- No filing or service","- No party or court contact",
    "- No external counsel instruction","- No settlement authority",
    "- No external legal advice","- No external distribution"
]
write_note(VAULT/"10_Reports"/"Baby_Step_4_Litigation_Committee_Memorandum.md",memo)


In [ ]:
dashboard = [
    "# Baby Step 4 — Litigation Committee Dashboard","",
    "## Portfolio status",""
]
for path,count in Counter(x["committee_path"] for x in committee_paths).items():
    dashboard.append(f"- {path}: **{count}**")
dashboard += ["","## Matter cards",""]
for item in committee_paths:
    rec = portfolio[item["matter_id"]]["recommendation"]
    dashboard += [
        f"### [[../02_Active_Matters/{item['matter_id']}]]","",
        f"- Recommendation: [[../08_Recommendations/{rec['recommendation_id']}]]",
        f"- Preferred strategy: {rec['preferred_strategy']}",
        f"- Permission: **{item['permission_state']}**",
        f"- Generalization: **{item['generalization_status']}**",
        f"- Average claim confidence: {item['average_claim_confidence']}/100",
        f"- Open contradictions: {item['open_contradictions']}",
        f"- Requested path: **{item['committee_path']}**",""
    ]
dashboard += ["## Decision request","","Approve each bounded internal path.","","No external action is requested."]
write_note(VAULT/"10_Reports"/"Baby_Step_4_Committee_Dashboard.md",dashboard)


In [ ]:
exhibits = []
for item in committee_paths:
    record = portfolio[item["matter_id"]]
    m = record["matter"]
    r = record["recommendation"]
    exhibits.append({
        "matter_id":item["matter_id"],
        "caption":m["caption"],
        "cause_of_action":m["cause_of_action"],
        "procedural_posture":m["procedural_posture"],
        "claimed_damages_usd":m["claimed_damages_usd"],
        "modeled_exposure_usd":m["modeled_exposure_usd"],
        "preferred_strategy":r["preferred_strategy"],
        "fallback_strategy":r["fallback_strategy"],
        "recommendation_confidence":r["confidence_score"],
        "average_claim_confidence":item["average_claim_confidence"],
        "permission_state":item["permission_state"],
        "generalization_status":item["generalization_status"],
        "open_contradictions":item["open_contradictions"],
        "committee_path":item["committee_path"]
    })

(VAULT/"data"/"baby_step_4_committee_exhibits.json").write_text(
    json.dumps(exhibits,indent=2),encoding="utf-8"
)
with (VAULT/"data"/"baby_step_4_committee_exhibits.csv").open("w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f,fieldnames=list(exhibits[0].keys()))
    w.writeheader()
    w.writerows(exhibits)


## Presentation deck

The notebook generates an editable PowerPoint directly into the vault.


In [ ]:
from pptx import Presentation
from pptx.util import Inches

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

def title_slide(title,subtitle):
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    slide.shapes.title.text = title
    slide.placeholders[1].text = subtitle

def bullets_slide(title,bullets):
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = title
    tf = slide.placeholders[1].text_frame
    tf.clear()
    for idx,bullet in enumerate(bullets):
        p = tf.paragraphs[0] if idx==0 else tf.add_paragraph()
        p.text = bullet

title_slide(
    "Corporate Civil Litigation Exo-Brain",
    "Baby Step 4 — First Litigation-Committee-Ready Product"
)
bullets_slide(
    "Governed Architecture",
    [
        "Knowledge plane: 1,000 synthetic precedents and five active matters",
        "Analysis plane: Recommendation V1 and matter-specific profiles",
        "Governance plane: sources, claims, confidence, contradictions, decisions",
        "Delivery plane: memorandum, dashboard, exhibits, presentation",
        "Human authority: no external action"
    ]
)
bullets_slide(
    "Portfolio Overview",
    [
        f"{x['matter_id']}: {x['committee_path']} | "
        f"permission={x['permission_state']} | "
        f"contradictions={x['open_contradictions']}"
        for x in committee_paths
    ]
)
for item in committee_paths:
    record = portfolio[item["matter_id"]]
    m = record["matter"]
    r = record["recommendation"]
    bullets_slide(
        f"{item['matter_id']} — {m['cause_of_action']}",
        [
            f"Preferred strategy: {r['preferred_strategy']}",
            f"Fallback: {r['fallback_strategy']}",
            f"Recommendation confidence: {r['confidence_score']}/100",
            f"Average claim confidence: {item['average_claim_confidence']}/100",
            f"Permission: {item['permission_state']}",
            f"Open contradictions: {item['open_contradictions']}",
            f"Requested path: {item['committee_path']}"
        ]
    )
bullets_slide(
    "Contradiction and Confidence View",
    [
        f"Atomic claims: {len(claims)}",
        f"Open contradictions: {len(contradictions)}",
        f"Average claim confidence: {statistics.mean(x['confidence_score'] for x in claims):.2f}/100",
        "High-severity contradictions restrict reliance",
        "Recommendation V1 remains preserved"
    ]
)
bullets_slide(
    "Requested Committee Decision",
    [
        "Approve each bounded internal committee path",
        "Permit controlled internal diligence and contradiction resolution",
        "Preserve Recommendation V1",
        "Do not create Recommendation V2",
        "No filing, contact, settlement, or external distribution"
    ]
)

deck = VAULT/"10_Reports"/"Baby_Step_4_Litigation_Committee_Deck.pptx"
prs.save(deck)
print("Slides:",len(prs.slides))


In [ ]:
DECISION = {
    "decision_id":"DEC-004",
    "date":datetime.date.today().isoformat(),
    "title":"Accept First Litigation-Committee Product",
    "decision":"Accept the Baby Step 4 memorandum, dashboard, exhibits, and presentation as the first committee-ready internal product.",
    "approved_paths":{x["matter_id"]:x["committee_path"] for x in committee_paths},
    "permitted_next_actions":[
        "controlled internal diligence",
        "evidence clarification",
        "contradiction resolution",
        "refresh of qualified claims",
        "preserve Recommendation V1"
    ],
    "not_authorized":[
        "Recommendation V2","filing","service","party contact","court contact",
        "external counsel instruction","settlement offer",
        "external legal advice","external distribution"
    ],
    "synthetic":True
}
(VAULT/"09_Decisions"/"DEC-004.json").write_text(json.dumps(DECISION,indent=2),encoding="utf-8")
lines = [
    "# DEC-004 — Accept First Litigation-Committee Product","",
    f"**Date:** {DECISION['date']}","","## Decision","",DECISION["decision"],"",
    "## Approved matter paths",""
]
lines += [f"- [[../02_Active_Matters/{mid}]] — **{path}**" for mid,path in DECISION["approved_paths"].items()]
lines += ["","## Permitted next actions",""]
lines += [f"- {x}" for x in DECISION["permitted_next_actions"]]
lines += ["","## Not authorized",""]
lines += [f"- {x}" for x in DECISION["not_authorized"]]
write_note(VAULT/"09_Decisions"/"DEC-004.md",lines)


In [ ]:
hot = [
    "# Current State — Hot Cache","",
    "## Recommendation state","",
    "- Recommendation V1 remains preserved.",
    "- Recommendation V2 does not exist.","",
    "## Committee product","",
    "- [[../10_Reports/Baby_Step_4_Litigation_Committee_Memorandum]]",
    "- [[../10_Reports/Baby_Step_4_Committee_Dashboard]]",
    "- `10_Reports/Baby_Step_4_Litigation_Committee_Deck.pptx`","",
    "## Approved matter paths",""
]
hot += [f"- {x['matter_id']}: **{x['committee_path']}**" for x in committee_paths]
hot += [
    "","## Current decision","","- [[../09_Decisions/DEC-004]]","",
    "## Next permitted experiment","",
    "Execute controlled diligence and refresh the evidence base."
]
write_note(VAULT/"12_Hot_Cache"/"Current_State.md",hot)


In [ ]:
errors = []
v1 = list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))
v2 = list((VAULT/"08_Recommendations").glob("REC-*-V002.md"))
if len(v1)!=5:
    errors.append(f"Expected 5 Recommendation V1 notes, found {len(v1)}")
if v2:
    errors.append("Recommendation V2 exists prematurely")
if len(committee_paths)!=5:
    errors.append("Not all matters received a committee path")

required = [
    VAULT/"10_Reports"/"Baby_Step_4_Litigation_Committee_Memorandum.md",
    VAULT/"10_Reports"/"Baby_Step_4_Committee_Dashboard.md",
    VAULT/"10_Reports"/"Baby_Step_4_Litigation_Committee_Deck.pptx",
    VAULT/"data"/"baby_step_4_committee_exhibits.json",
    VAULT/"data"/"baby_step_4_committee_exhibits.csv",
    VAULT/"data"/"baby_step_4_committee_paths.json",
    VAULT/"09_Decisions"/"DEC-004.md",
    VAULT/"09_Decisions"/"DEC-004.json"
]
for p in required:
    if not p.exists():
        errors.append(f"Missing: {p}")

validation = {
    "validated_at":datetime.datetime.now().isoformat(),
    "committee_path_count":len(committee_paths),
    "recommendation_v1_count":len(v1),
    "recommendation_v2_count":len(v2),
    "decision":"DEC-004",
    "errors":errors,
    "passed":len(errors)==0
}
(VAULT/"11_Audit"/"Baby_Step_4_Validation.json").write_text(
    json.dumps(validation,indent=2),encoding="utf-8"
)
assert validation["passed"],errors
print(json.dumps(validation,indent=2))
print("BABY STEP 4 PASSED")


In [ ]:
state.update({
    "completed_steps":sorted(set(state.get("completed_steps",[])+[4])),
    "current_step":4,
    "next_step":5,
    "decision":"DEC-004",
    "committee_product_created":True,
    "recommendation_count":5,
    "current_recommendation_version":1,
    "next_problem":"Execute controlled diligence and refresh the evidence base.",
    "permission_state":{
        "observe":True,
        "organize":True,
        "browse":True,
        "internal_strategy_analysis":True,
        "evidence_governance":True,
        "committee_product":True,
        "controlled_internal_diligence":True,
        "recommendation_v2":False,
        "external_action":False
    }
})
state_path.write_text(json.dumps(state,indent=2),encoding="utf-8")
audit = {
    "timestamp":datetime.datetime.now().isoformat(),
    "step":4,
    "action":"Produced first litigation-committee-ready memorandum, dashboard, exhibits, and presentation.",
    "outputs":{"committee_paths":len(committee_paths),"memorandum":True,"presentation":True,"decision":"DEC-004"},
    "validation_passed":True
}
with (VAULT/"11_Audit"/"workflow_audit.jsonl").open("a",encoding="utf-8") as f:
    f.write(json.dumps(audit)+"\n")
